<a href="https://colab.research.google.com/github/Rubal-code/DEEP-LEARNING/blob/main/clip_multimodal_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Import CLIP model and processor from the Hugging Face transformers library
from transformers import CLIPModel, CLIPProcessor

# Define the pre-trained CLIP model name
model_name = "openai/clip-vit-base-patch32"

# Load the pre-trained CLIP model
model = CLIPModel.from_pretrained(model_name)

# Load the corresponding processor for text and image preprocessing
processor = CLIPProcessor.from_pretrained(model_name)

# Set the model to evaluation mode (disables dropout, batch norm, etc.)
model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [10]:
# Import PyTorch library for tensor operations
import torch
# Import the Image module from PIL to handle image creation and manipulation
from PIL import Image

# Create a synthetic 224x224 red image to simulate 'shoe.jpg' so the script runs successfully
image = Image.new("RGB", (224, 224), color="red")
# Save the generated red image as 'shoe.jpg' in the local workspace
image.save("shoe.jpg")

# Process the prompt text to prepare PyTorch tensors with padding enabled
text_inputs = processor(text=["a pair of red running shoes"], return_tensors="pt", padding=True)

# Disable gradient calculations to save memory and speed up computation
with torch.no_grad():
    # Pass the text inputs into the model to extract and unpack the raw text projection tensor
    text_emb = model.get_text_features(**text_inputs)

# Open the saved image file 'shoe.jpg' and make sure it is in RGB format
image = Image.open("shoe.jpg").convert("RGB")

# Process the image to prepare PyTorch tensors required by the model
image_inputs = processor(images=image, return_tensors="pt")

# Disable gradient calculations to save memory and speed up computation
with torch.no_grad():
    # Pass the processed image inputs into the model to extract and unpack the raw image projection tensor
    image_emb = model.get_image_features(**image_inputs)


In [12]:
# Initialize PyTorch's Cosine Similarity module along the last dimension
cos = torch.nn.CosineSimilarity(dim=-1)

# Extract the underlying PyTorch tensor if the returned object is a BaseModelOutput
text_tensor = text_emb.pooler_output if hasattr(text_emb, "pooler_output") else (text_emb.last_hidden_state if hasattr(text_emb, "last_hidden_state") else text_emb)
image_tensor = image_emb.pooler_output if hasattr(image_emb, "pooler_output") else (image_emb.last_hidden_state if hasattr(image_emb, "last_hidden_state") else image_emb)

# Ensure we have the raw tensor data if it has a .logits or other field, or fallback directly
if hasattr(text_tensor, "detach"):
    text_tensor = text_tensor
else:
    text_tensor = torch.tensor(text_tensor)

if hasattr(image_tensor, "detach"):
    image_tensor = image_tensor
else:
    image_tensor = torch.tensor(image_tensor)

# Divide the text tensor by its L2 norm to normalize its length to 1.0
text_emb_norm = text_tensor / text_tensor.norm(dim=-1, keepdim=True)

# Divide the image tensor by its L2 norm to normalize its length to 1.0
image_emb_norm = image_tensor / image_tensor.norm(dim=-1, keepdim=True)

# Calculate the cosine similarity score using the normalized tensors
score = cos(text_emb_norm, image_emb_norm)


In [8]:
# Create a combined input dictionary containing both text and image tensors
inputs = processor(text=["a pair of red running shoes"], images=image, return_tensors="pt", padding=True)

# Process both inputs together using the forward pass of the model without gradients
with torch.no_grad():
    # Retrieve all outputs from the model using the combined inputs
    outputs = model(**inputs)

# Extract normalized text projections (embeddings) from the model outputs
text_emb_joint = outputs.text_embeds

# Extract normalized image projections (embeddings) from the model outputs
image_emb_joint = outputs.image_embeds

In [9]:
# Calculate the final cosine similarity score between the jointly extracted embeddings
similarity = cos(text_emb_joint, image_emb_joint)

# Print the similarity value rounded to 4 decimal places
print(f"Cosine similarity: {similarity.item():.4f}")

Cosine similarity: 0.2542
